# 5-fold CV: orig / clean

Scenario 1 from the draft. Outer `StratifiedKFold(n=5, seed=42)` holds out 20% as test. From the remaining 80%, a stratified 1/8 split is val, so each fold is **70/10/20** of that variant's pool. A 5-fold cannot be 80/10/10 because test is 20%.

Per-fold seed is `42 + fold`. Split membership: `results/splits/{variant}-fold{k}.csv`. Fold weights are not saved.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Markdown

HERE = Path.cwd()
if HERE.name != "cnn-latest":
    HERE = Path("training/notebooks/cnn-latest").resolve()
KFOLD_CSV = HERE / "results" / "kfold.csv"
VARIANTS = ["orig", "clean"]
FOLDS = list(range(5))

def load_kfold():
    if KFOLD_CSV.exists():
        return pd.read_csv(KFOLD_CSV)
    return pd.DataFrame(columns=["variant", "fold"])

jobs = [(v, f) for v in VARIANTS for f in FOLDS]
done = load_kfold()
pending = [(v, f) for v, f in jobs if done.empty or not ((done.variant == v) & (done.fold == f)).any()]
display(Markdown(f"Pending **{len(pending)}** / {len(jobs)}"))

for variant, fold in pending:
    display(Markdown(f"### `{variant}` fold {fold}"))
    result = subprocess.run(
        ["uv", "run", "python", "run_kfold.py", "--variant", variant, "--fold", str(fold)],
        cwd=HERE,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"{variant}/fold{fold} exited {result.returncode}")

kf = load_kfold().sort_values(["variant", "fold"])
display(kf.reset_index(drop=True))
if len(kf):
    display(kf.groupby("variant")[["test_accuracy", "test_macro_f1"]].agg(["mean", "std"]).round(4))